In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from sklearn.metrics import root_mean_squared_error as root_mse
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
import plotly.express as px
import seaborn as sns
from pymatgen.core import Structure
from mx2che_tools import getFormula2Compare2DMD, elements, encodeRGB, getFormula2CompareFFCCS, T_, T_delta_Sconf, structShifter, getDescriptors

### Mapping 2DMD data

In [ ]:
# Searching for 2dmd structures with high defect contents
# Get 5 formulas for additional evaluation

path_2dmd = 'data/2d-materials-point-defects-all-table.pkl.gz'
data_2dmd = pd.read_pickle(path_2dmd)
data_2dmd = data_2dmd[(data_2dmd['base'] == 'WSe2') | (data_2dmd['base'] == 'MoS2')].copy()
data_2dmd['natoms'] = data_2dmd['atomic_numbers'].apply(len)
data_2dmd['formation_energy_per_atom'] = data_2dmd['formation_energy'] / data_2dmd['natoms']

energies_df = pd.read_pickle('data/energies_df.pkl.gz')
missed_data = [col for col in energies_df.columns if col not in data_2dmd.columns]
data_2dmd_energies = pd.concat((data_2dmd, energies_df[energies_df['part'] == '2DMD datasets'][missed_data]), axis=1)
hdc_2dmd_energies = data_2dmd_energies[data_2dmd_energies['defect_concentration'] == 'high'].copy()
hdc_2dmd_energies['formula2compare'] = [getFormula2Compare2DMD(hdc_2dmd_energies.loc[i]) for i in hdc_2dmd_energies.index]
hdc_2dmd_energies['formula2compare.25'] = [getFormula2Compare2DMD(hdc_2dmd_energies.loc[i], multiplier=0.25) for i in hdc_2dmd_energies.index]
elements_df = pd.DataFrame(np.vstack(hdc_2dmd_energies['formula2compare'].apply(lambda x: np.array([int(el) for el in re.split('\D', x) 
                                                                                                    if el != ''])).values), 
                           columns=elements, index=hdc_2dmd_energies.index)
for col in elements_df.columns:
    elements_df[f'{col}_res'] = elements_df[col] % 4    
elements_df['res_sum'] = elements_df.iloc[:, -len(elements):].sum(axis=1).values

hdc_2dmd_energies = pd.concat((hdc_2dmd_energies, elements_df), axis=1)
formula_to_compare = hdc_2dmd_energies[hdc_2dmd_energies['res_sum'] == 0]['formula2compare.25'].unique()
hdc_2dmd_energies[hdc_2dmd_energies['res_sum'] == 0].groupby(['base', 'defect_nsites', 'space_group_no',
                                                            'formula2compare.25']).size()

In [ ]:
# Statistics of the 4x4x1 supercell-based CCS
print('formula2compare: ', formula_to_compare)
CCS441_statistics = pd.read_pickle('data/comp_num.pkl.gz')
print('structures mapped from 2DMD to 4x4x1 supercell\n', CCS441_statistics[np.isin(CCS441_statistics['composition'], formula_to_compare)])
print('total count of structures after merge', CCS441_statistics['num_struct_after'].sum())
print('missed in the 4x4x1 CCS by design:', formula_to_compare[~np.isin(formula_to_compare, CCS441_statistics['composition'])])

### 5formulas CCS: structures and energies

In [ ]:
base2form_dict = {'MoS2': 'Mo14S28Se2W1', 'WSe2': 'Mo1S2Se28W14'}
form2form_dict = {'W1Mo14U1Se2S28Np2': 'Mo14S28Se2W1', 'W14Mo1U1Se28S2Np2': 'Mo1S2Se28W14',
                 'Mo56S112Se8W4': 'Mo14S28Se2W1', 'Mo4S8Se112W56': 'Mo1S2Se28W14'}

In [ ]:
ff_ccs_inference = pd.read_pickle("data/5formula_ccs_inference.pkl.gz")
ff_ccs_inference.groupby(['DFT-derived', 'Formula', 'Space_group_no', 'Space_group_symbol']).size()

In [ ]:
ff_available_models = ff_ccs_inference.columns[-24:]
for model in ff_available_models:
    fig, axs = plt.subplots(nrows=1, ncols=2,  figsize=(8, 4), sharey=True)
    fig.suptitle(f'{model}', y=0.91)
    axs[0].set_ylabel('Number of structures')
    for formula_ind, formula in enumerate(['W1Mo14U1Se2S28Np2', 'W14Mo1U1Se28S2Np2']):
        axs[formula_ind].set_title(re.sub('[\d]', lambda x: f'$_{x.group()}$', form2form_dict[formula]))
        axs[formula_ind].hist([ff_ccs_inference[(ff_ccs_inference['DFT-derived'] == DFTderived) & (ff_ccs_inference['Formula'] == formula)][model] 
                               for DFTderived in [True, False]],
                              label=[f'{sym}' for sym in ['$C2-Amm2$', '$P1$']], bins=25, edgecolor='grey', stacked=True)
        axs[formula_ind].set_xlabel('GNN: $E_{F}$, eV/atom')
        axs[formula_ind].grid(alpha=0.3)
        axs[formula_ind].set_yscale('log')
    axs[0].set_ylim((1e0, 1e6))
    axs[0].legend(title='Space groups', labelspacing=0.01)
    plt.tight_layout()
    plt.show()

In [ ]:
modes_all = ['2dmd-ldc', '2dmd-hdc', '2dmd', 'ccs-Pmm2+', 'ccs-C2+', 'ccs']
mode_color = dict(zip(modes_all, ['b', 'g', 'r', 'c', 'm', 'grey']))
target_energy_2dmd = 'structure_formation_energy_per_atom'
fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(10, 6), sharex=True, sharey=True)
for archit_id, archit in enumerate(['allegro', 'nequip']):
    for pretrain_id, pretrain in enumerate(['aflow', 'random']):
        axs[1, pretrain_id].set_xlabel('$E_{F}$, eV/atom')
        axs[archit_id, 0].set_ylabel('Number of configurations')

        ff_available_models_of_archit = [f'{archit}_{pretrain}_{mode}' for mode in modes_all]
        axs[archit_id, pretrain_id].set_title(f'{archit}_{pretrain}')
        axs[archit_id, pretrain_id].hist([ff_ccs_inference[model] for model in ff_available_models_of_archit], 
                                         stacked=True, bins=40, edgecolor='grey', label=modes_all, color=list(map(lambda x: mode_color[x], modes_all)))
        axs[archit_id, pretrain_id].grid(alpha=0.3)
        
for formula in ['W1Mo14U1Se2S28Np2', 'W14Mo1U1Se28S2Np2']:
    x = hdc_2dmd_energies[hdc_2dmd_energies['formula2compare.25'] == formula][target_energy_2dmd]
    axs[0, 0].text(x.min() - 0.03, 1.4e5, re.sub('[\d]', lambda x: f'$_{x.group()}$', form2form_dict[formula])+' (2DMD)',
                   color='k', alpha=0.8, rotation=90)
    for archit_id, _ in enumerate(['allegro', 'nequip']):
        for pretrain_id, _ in enumerate(['aflow', 'random']):
            axs[archit_id, pretrain_id].axvline(x=x.min(), color='k', alpha=0.6, lw=1)
            axs[archit_id, pretrain_id].axvline(x=x.max(), color='k', alpha=0.6, lw=1)            

axs[archit_id, pretrain_id].legend(title='Training data', bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()

In [ ]:
# Read DFT-derived properties for HS structures of 5 formula CCS and split into 1 and 6 defect subsets
structures_Ndef_result = pd.read_pickle('data/structures_Ndef.pkl.gz')
structures_1def_result = structures_Ndef_result[structures_Ndef_result['nelements'] != 4].copy()
structures_6def_result = structures_Ndef_result[structures_Ndef_result['nelements'] == 4].copy()

# Read results of recalculations for 2dms lowest and highest energies
structures_24def_result = pd.read_pickle('data/structures_24def.pkl.gz')

In [ ]:
# Comparison of 6 defs subset of ff_ccs with 24 defs structures of the 2DMD dataset
plt.figure(figsize=(11, 3.5))
plt.hist([data_2dmd_energies[(data_2dmd_energies['defect_nsites']==24)]['structure_formation_energy_per_atom'], 
          structures_6def_result['formation_energy_per_atom']], 
         bins=np.linspace(-.88, -.36, 105),  # 5 meV/atom bins
         edgecolor='grey', stacked=True, color=[encodeRGB(px.colors.qualitative.Bold[3]), 'darkgrey'],
         label=['2DMD\n24 defects (8x8x1 supercell)\n$P1$ space group', 'First enhanced CCS\n6 defects (4x4x1 supercell)\n$C2-Amm2$ space groups'])
for base in ['MoS2', 'WSe2']:
    ref_ehull = data_2dmd_energies[(data_2dmd_energies['base'] == base) & (data_2dmd_energies['defect_nsites'] == 24)]['Reference_hull_E'].unique()
    formula = base2form_dict[base]
    formula_tag = re.sub('[\d]', lambda x: f'$_{x.group()}$', formula)
    plt.scatter(ref_ehull[0], 1, color='k', marker='v')
    plt.text(ref_ehull[0], 1.3, f'0 K convex hull\nof {formula_tag}\ncomposition', horizontalalignment='center', verticalalignment='bottom')
    plt.plot(sorted(structures_24def_result[structures_24def_result['base']==base]['formation_energy_per_atom']), [1, 1], lw=5, color='gold')
title = ('\t'*10).join([re.sub('[\d]', lambda x: f'$_{x.group()}$', base2form_dict[base]) for base in ['MoS2', 'WSe2']])
plt.title(title)
plt.legend(labelspacing=0.1)
plt.yscale('log')
plt.ylabel('Number of entries')
plt.grid(alpha=0.3)
plt.xlabel('DFT-derived $E_{F}$, eV/atom')
plt.tight_layout()
plt.show()

### HT1 in general (paper) and for each base material

In [ ]:
# Merge dft results, ff_ccs metainformation, and inference results
ff_ccs_inference_dft = pd.concat((ff_ccs_inference, structures_Ndef_result.drop('natoms', axis=1)), axis=1)
ff_ccs_inference_dft['formula_string'] = [getFormula2CompareFFCCS(ff_ccs_inference_dft.loc[i], ['Mo', 'S', 'Se', 'W'])
                                          for i in ff_ccs_inference_dft.index]
print(ff_ccs_inference_dft.shape)
print(ff_ccs_inference_dft.columns)
ff_ccs_inference_dft.groupby(['formula_string', 'natoms']).size()

In [ ]:
plt.figure(figsize=(5, 5))
for model in ['nequip_aflow_2dmd-ldc', 'nequip_random_2dmd-hdc', 'allegro_random_2dmd-ldc']:
    plt.scatter(ff_ccs_inference_dft[model], ff_ccs_inference_dft['formation_energy_per_atom'], label=model, alpha=0.5, s=10)
plt.plot([-.9, -.3], [-.9, -.3], 'k', alpha=0.3, label='perfect prediction')
plt.legend(labelspacing=0.1)  # bbox_to_anchor=(1, 1)
plt.grid(alpha=0.3)
plt.ylabel('DFT-derived FEs, eV/atom')
plt.xlabel('GNN-assessed FEs, eV/atom')
plt.show()

In [ ]:
ff_ccs_inference_dft_45 = ff_ccs_inference_dft.dropna()
ff_ccs_inference_dft_45 = ff_ccs_inference_dft_45[ff_ccs_inference_dft_45['natoms'] == 45].copy()

fig, axs = plt.subplots(nrows=4, ncols=6, figsize=(15, 11), sharex=True, sharey=True)
for arch_index, arch_ in enumerate(['allegro', 'nequip']):    
    for pr_index_, pretrain_set_ in enumerate(['random', 'aflow']):        
        axs[2 * arch_index + pr_index_, 0].set_ylabel('DFT-derived $E_{F}$, eV/atom')        
        for tr_index_, train_set_ in enumerate(['2dmd', '2dmd-hdc', '2dmd-ldc', 'ccs', 'ccs-C2+', 'ccs-Pmm2+']):
            model_ = f'{arch_}_{pretrain_set_}_{train_set_}'
            axs[2 * arch_index + pr_index_, tr_index_].set_title(model_)
            axs[2 * arch_index + pr_index_, tr_index_].scatter(ff_ccs_inference_dft_45[model_], ff_ccs_inference_dft_45['formation_energy_per_atom'],
                                                               alpha=0.2)
            axs[2 * arch_index + pr_index_, tr_index_].plot([-.8, -.3], [-.8, -.3], 'k', alpha=0.3)
            axs[2 * arch_index + pr_index_, tr_index_].grid(alpha=0.3)

for tr_index_, train_set_ in enumerate(['2dmd', '2dmd-hdc', '2dmd-ldc', 'ccs', 'ccs-C2+', 'cc+Pmm2+']):
    axs[-1, tr_index_].set_xlabel('GNN-predicted $E_{F}$, eV/atom')
plt.tight_layout()
plt.show()

scores = pd.DataFrame([(model, 1000 * root_mse(ff_ccs_inference_dft_45['formation_energy_per_atom'], ff_ccs_inference_dft_45[model]))
                       for model in ff_available_models], columns=['model', 'rmse,meV/atom'])
scores['architecture'] = scores['model'].apply(lambda x: 'Allegro\n' if x.startswith('allegro') else 'NequIP\n')
scores['pre-training'] = scores['model'].apply(lambda x: re.match('.*random.*', x) == None)
scores['pre-training data'] = scores['pre-training'].apply(lambda x: '  (AflowLib)' if x else '')
scores['data used'] = scores['model'].apply(lambda x: x.split('random_')[1] if re.match('.*random.*', x) != None else x.split('aflow_')[1])
scores['model'] = scores['architecture'] + scores['pre-training data']
scores = scores[sorted(scores.columns)]
scores = scores[scores['data used'] != 'ccs_more_than_half_Me_tt']
scores['data used'] = scores['data used'].apply(lambda x: x if x != 'ccs_more_than_half_Me' else 'ccs')

plt.figure(figsize=(4.8, 3.98))
ax = sns.heatmap(scores.pivot(index='model', columns='data used', values='rmse,meV/atom'), vmin=5 , vmax=65, annot=True, fmt=".1f",
                 cbar_kws={'label': 'RMSE, meV/atom', 'drawedges': False, 'shrink': 0.9})
ax.set(xlabel='Training set', ylabel='Architecture (pretraining)')
ax.set_title('HT#1 (5556 structures)')
plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=2, sharey=True, figsize=(7.5, 4))
cbar_ax = fig.add_axes([1, 0.35, .02, .5])

for form_ind_ , formula_string in enumerate(list(map(lambda x: base2form_dict[x], ['MoS2', 'WSe2'])) ):
    ff_ccs_inference_dft_base = ff_ccs_inference_dft[ff_ccs_inference_dft['formula_string'] == formula_string].dropna().copy()
    scores = pd.DataFrame([(model, 1000 * root_mse(ff_ccs_inference_dft_base['formation_energy_per_atom'], ff_ccs_inference_dft_base[model]))
                            for model in ff_available_models], columns=['model', 'rmse,meV/atom'])
    scores['architecture'] = scores['model'].apply(lambda x: 'Allegro\n' if x.startswith('allegro') else 'NequIP\n')
    scores['pre-training'] = scores['model'].apply(lambda x: re.match('.*random.*', x) == None)
    scores['pre-training data'] = scores['pre-training'].apply(lambda x: '    (AflowLib)' if x else '')
    scores['data used'] = scores['model'].apply(lambda x: x.split('random_')[1] if re.match('.*random.*', x) != None else x.split('aflow_')[1])
    scores['model'] = scores['architecture'] + scores['pre-training data']
    scores = scores[sorted(scores.columns)]
    scores = scores[scores['data used'] != 'ccs_more_than_half_Me_tt']
    scores['data used'] = scores['data used'].apply(lambda x: x if x != 'ccs_more_than_half_Me' else 'ccs')

    if form_ind_ != 0:
        sns.heatmap(scores.pivot(index='model', columns='data used', values='rmse,meV/atom'),
                    ax=axs[form_ind_], vmin=5 , vmax=65, annot=True, fmt=".1f",
                    cbar_ax=cbar_ax,
                    cbar_kws={'label': 'RMSE, meV/atom', 'drawedges': False, 'shrink': 0.9})
    else:
        sns.heatmap(scores.pivot(index='model', columns='data used', values='rmse,meV/atom'),
                    ax=axs[form_ind_], vmin=5, vmax=65, annot=True, fmt=".1f", cbar=False)
    
    axs[form_ind_].set_title(re.sub('[\d]', lambda x: f'$_{x.group()}$', formula_string) + f'({ff_ccs_inference_dft_base.shape[0]} structures)')
axs[0].set(xlabel='Training set', ylabel='Architecture (pretraining)')
axs[1].set(xlabel='Training set', ylabel=None)
plt.tight_layout()
plt.show()

### Configurational entropy impact on decomposition energy gap

In [ ]:
space_group_lim_sets = {0: ('$P1-Amm2$', '-', 'g'), 1: ('$C2-Amm2$', '--', 'k')}

# Set list of models
for model in ff_available_models[[1, 2, 7]]: ## top-3 models on HT#1
    fig, axs = plt.subplots(nrows=1, ncols=2, sharey=True, figsize=(6.5, 4))
    for i, base in enumerate(['MoS2', 'WSe2']):
        # 2DMD part
        ref_ehull = data_2dmd_energies[(data_2dmd_energies['base'] == base) & (data_2dmd_energies['defect_nsites'] == 24)]['Reference_hull_E'].unique()
        formula = data_2dmd_energies[(data_2dmd_energies['base'] == base) & (data_2dmd_energies['defect_nsites'] == 24)]['formula_str'].unique()
        assert (len(ref_ehull) == 1) and (len(formula) == 1)
        ref_ehull, formula = ref_ehull[0], formula[0]
        subset_2dmd = data_2dmd_energies[(data_2dmd_energies['base'] == base) & (data_2dmd_energies['defect_nsites'] == 24)]
        energies_per_cell = subset_2dmd['structure_formation_energy']
        weights = 768 * np.ones_like(energies_per_cell)  # estimated weights of P1 structures
        structures_available = len(weights)
        natom_  = subset_2dmd['natoms'].unique()[0]
        axs[i].plot(T_, [1000 * (T_delta_Sconf(energies_per_cell, weights, t, natoms_for_per_atom=natom_) - ref_ehull) for t in T_], 
                    label=f'DFT (2DMD): {structures_available} strs.\n$P1$ s.g.', color=encodeRGB(px.colors.qualitative.Bold[3]), lw=2.5, alpha=0.8)

        # DFT-derived part
        subset_6def = ff_ccs_inference_dft[ff_ccs_inference_dft['formula_string'] == base2form_dict[base]]
        subset_6def_dft = subset_6def.dropna()  # drop elements not available from dft
        energies_per_atom = subset_6def_dft['formation_energy_per_atom']
        energies_per_cell = energies_per_atom * subset_6def_dft['natoms']
        weights=subset_6def_dft['Weight'].values
        structures_available = len(weights)
        natom_ =subset_6def_dft['natoms'].unique()[0]
        axs[i].plot(T_, [1000 * (T_delta_Sconf(energies_per_cell, weights, t, natoms_for_per_atom=natom_) - ref_ehull) for t in T_],
                    label=f'DFT: {structures_available} strs.\n$C2-Amm2$ s.g.\'s', color='darkgrey', lw=2.5, alpha=0.8)
        
        # GNN predicted part
        for space_group_limit in [1, 0]:  # set space group limit for conf entr
            sgg, sgs, sgc  = space_group_lim_sets[space_group_limit]
            subset_6def_gnn = subset_6def[subset_6def['Space_group_no']>space_group_limit]
            energies_per_atom_model = subset_6def_gnn[model]
            energies_per_atom_dft = subset_6def_gnn['formation_energy_per_atom']
            if space_group_limit == 1:
                rmse = 1000 * root_mse(energies_per_atom_dft.dropna(), energies_per_atom_model.loc[energies_per_atom_dft.dropna().index])
                print(f'{base} ({sgg}): {model}\'s RMSE = {rmse:.2f} meV/atom')
            energies_per_cell_model = energies_per_atom_model * subset_6def_gnn['natoms']
            weights=subset_6def_gnn['Weight'].values
            structures_available = len(weights)
            natom_ =subset_6def_gnn['natoms'].unique()[0]
            deltaG_ = [1000 * (T_delta_Sconf(energies_per_cell_model, weights, t, natoms_for_per_atom=natom_) - ref_ehull) for t in T_]
            axs[i].plot(T_, deltaG_, sgs, color=sgc, lw=0.7, label=f'GNN: {structures_available} strs.\n{sgg} s.g.\'s')
            print('∆Gdrop ', len(weights), deltaG_[0] - deltaG_[-1])
            if space_group_limit == 0:  # doing the same for favored P1+ structures only
                sgc = 'green'
                sgs = ':'
                for lw_value, str_num_limit in enumerate([100, 500, 2500]):
                    subset_6def_gnn_fav = subset_6def_gnn.sort_values(model).iloc[:str_num_limit, :]
                    energies_per_atom_model = subset_6def_gnn_fav[model]
                    energies_per_cell_model = energies_per_atom_model * subset_6def_gnn_fav['natoms']
                    weights=subset_6def_gnn_fav['Weight'].values
                    structures_available = len(weights)
                    natom_ =subset_6def_gnn_fav['natoms'].unique()[0]
                    axs[i].plot(T_, [1000 * (T_delta_Sconf(energies_per_cell_model, weights, t, natoms_for_per_atom=natom_) - ref_ehull) for t in T_],
                                sgs, color=sgc, lw=2 * (lw_value+.5), alpha=0.8, label=f'GNN: {structures_available} most fav. strs.')
        axs[i].set_title(re.sub('[\d]', lambda x: f'$_{x.group()}$', form2form_dict[formula]))
        axs[i].grid(alpha=0.3)
        print(ref_ehull)
        axs[i].set_xlabel('T, K')
    axs[0].set_ylabel('${\Delta}G$ with $T{\Delta}S_{conf}$\nabove 0 K hull, meV/atom')
    plt.legend(bbox_to_anchor=(1,1))
    plt.tight_layout()
    plt.show()

### Roots of energetical favorability

In [ ]:
rr_dft = GridSearchCV(estimator=Ridge(random_state=0), n_jobs=-1, param_grid={'alpha':np.linspace(0.1, 1, 10)})
rr_gnn = GridSearchCV(estimator=Ridge(random_state=0), n_jobs=-1, param_grid={'alpha':np.linspace(0.1, 1, 10)})
rfr_dft = GridSearchCV(estimator=RandomForestRegressor(n_jobs=-1, random_state=0), param_grid={'n_estimators':np.linspace(100, 500, 5, dtype=int)})
rfr_gnn = GridSearchCV(estimator=RandomForestRegressor(n_jobs=-1, random_state=0), param_grid={'n_estimators':np.linspace(100, 500, 5, dtype=int)})

# Top-3 models on HT#1
models = ['allegro_random_2dmd-ldc',  #rmse = 10.9 meV/atom 
          'nequip_aflow_2dmd-ldc',    #rmse = 11.2 meV/atom
          'nequip_random_2dmd-hdc']   #rmse = 13.0 meV/atom
model = models[0]

#### Mo14Se2S28W1

In [ ]:
formula = 'Mo14S28Se2W1'
base_structures = ff_ccs_inference_dft[(ff_ccs_inference_dft['formula_string'] == formula)].copy()
base_structures.dropna(inplace=True)
base_structures['favorability'] = base_structures['formation_energy_per_atom'] < -0.73
base_structures['cell_a'] = base_structures['initial_structure'].apply(lambda x: x.lattice.a)
base_structures['cell_b'] = base_structures['initial_structure'].apply(lambda x: x.lattice.b)
base_structures['cell_c'] = base_structures['initial_structure'].apply(lambda x: x.lattice.c)
base_structures['initial_structure_marked'] = base_structures['CIF_data'].apply(lambda x: Structure.from_str(input_string=x, fmt='cif'))
base_structures['initial_structure_marked'] = base_structures['initial_structure_marked'].apply(structShifter)

for parameter in ['a', 'b', 'c']:
    unique_par = base_structures[f'cell_{parameter}'].unique()
    print(f'unique parameter {parameter} is' , unique_par)

plt.hist([base_structures[base_structures['favorability'] == fav]['formation_energy_per_atom'] for fav in [True, False]], 
         color=['g', 'r'], stacked=True, edgecolor='k', 
         bins=np.linspace(-.9, -.6, 61), label=['favored', 'unfavored'])  # 5 meV/atom bins
plt.yscale('log')
plt.grid(alpha=0.3)
plt.xlabel('DFT-derived formation energy, eV/atom')
plt.ylabel('Number of structures')
plt.show()

print('energy gap position:')
min_unfav_value = base_structures[base_structures['favorability'] == False]['formation_energy_per_atom'].min()
max_fav_value = base_structures[base_structures['favorability'] == True]['formation_energy_per_atom'].max()
print(np.mean((min_unfav_value, max_fav_value)))

In [ ]:
# Descriptors
distrib = getDescriptors(base_structures,
                         elements_list_dict={'U': ['S', 'Se', 'Np', 'Mo', 'W'], 
                                              'W': ['S', 'Se', 'Np', 'Mo', 'U']},
                         elements_dict={'U': '$V_{Me}$', 'Mo': 'Mo', 'W': 'W',
                                        'Np': '$V_{X}$', 'S': 'S',  'Se': 'Se'},
                         normalize_over_cs=False)

# Print report on descriptors the heatmap with the mask and correct aspect ratio
print('distrib\'s shape', distrib.shape)
print(distrib.groupby(['FE ranges, eV/atom', 'favorability']).size())
col_subset = sorted(distrib.columns[:-5])
print('feature set: ', col_subset)
print('inequivalent features: ', len(col_subset))
print('unique sets among distrib\'s entries', sum(~distrib.duplicated(subset=col_subset)))
corr = distrib[col_subset].corr()

# Draw the heatmap with the mask and correct aspect ratio
plt.figure(figsize=(6, 6))
sns.heatmap(corr, center=0, cmap='vlag', square=True, linewidths=.5, cbar_kws={"shrink": .5})
plt.show()

# Fit descriptor-based models
X = distrib[col_subset]
y = distrib['formation_energy_per_atom']
y_model = distrib[model]
rr_dft.fit(X, y)
rr_gnn.fit(X, y_model)
rfr_dft.fit(X, y)
rfr_gnn.fit(X, y_model)

In [ ]:
print('**** DFT results ****')
print(f'Ridge Regression: RMSE = {1000 * root_mse(y, rr_dft.predict(X)):.2f} meV/atom')
print(f'Random Forest:    RMSE = {1000 * root_mse(y, rfr_dft.predict(X)):.2f} meV/atom')
print('\n**** GNN predictions ****')
print(f'Ridge Regression: RMSE = {1000 * root_mse(y, rr_gnn.predict(X)):.2f} meV/atom')
print(f'Random Forest:    RMSE = {1000 * root_mse(y, rfr_gnn.predict(X)):.2f} meV/atom')

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=2, sharey=True, figsize=(6, 3.5))
fig.suptitle(re.sub('[\d]', lambda x: f'$_{x.group()}$', formula), y=0.94)
axs[0].scatter(rr_dft.best_estimator_.coef_, range(len(col_subset)), alpha=0.7)
axs[0].scatter(rr_gnn.best_estimator_.coef_, range(len(col_subset)), alpha=0.5)
axs[1].barh(range(len(col_subset)), rfr_dft.best_estimator_.feature_importances_, height=0.4, align='edge', alpha=0.7, label='DFT')
axs[1].barh(range(len(col_subset)), rfr_gnn.best_estimator_.feature_importances_, height=-0.4, align='edge', alpha=0.4, label=f'GNN')
axs[0].set_xlim((-.03, .03))
axs[0].set_xlabel('RR coefficients, eV/pair')
axs[1].set_xscale('log')
axs[1].set_xlabel('RF feature importance, a.u.')
axs[1].set_xlim((5e-3, 1))
axs[1].legend(title='$E_{F}$ values')

for i in range(2):
    axs[i].grid(alpha=0.3)
plt.yticks(list(range(len(col_subset))), col_subset)
plt.tight_layout()
plt.show()

#### Mo1Se28S2W14

In [ ]:
formula = 'Mo1S2Se28W14'
base_structures = ff_ccs_inference_dft[(ff_ccs_inference_dft['formula_string'] == formula)].copy()
base_structures.dropna(inplace=True)
base_structures['favorability'] = base_structures['formation_energy_per_atom'] < -0.44
base_structures['cell_a'] = base_structures['initial_structure'].apply(lambda x: x.lattice.a)
base_structures['cell_b'] = base_structures['initial_structure'].apply(lambda x: x.lattice.b)
base_structures['cell_c'] = base_structures['initial_structure'].apply(lambda x: x.lattice.c)
base_structures['initial_structure_marked'] = base_structures['CIF_data'].apply(lambda x: Structure.from_str(input_string=x, fmt='cif'))
base_structures['initial_structure_marked'] = base_structures['initial_structure_marked'].apply(structShifter)

for parameter in ['a', 'b', 'c']:
    unique_par = base_structures[f'cell_{parameter}'].unique()
    print(f'unique parameter {parameter} is' , unique_par)
    
plt.hist([base_structures[base_structures['favorability'] == fav]['formation_energy_per_atom'] 
          for fav in [True, False]], color=['g', 'r'], stacked=True, edgecolor='k', 
         bins=np.linspace(-.6, -.3, 61), label=['favored', 'unfavored'])  # 5 meV/atom bins
plt.yscale('log')
plt.grid(alpha=0.3)
plt.xlabel('DFT-derived formation energy, eV/atom')
plt.ylabel('Number of structures')
plt.show()

print('energy gap position:')
min_unfav_value = base_structures[base_structures['favorability'] == False]['formation_energy_per_atom'].min()
max_fav_value = base_structures[base_structures['favorability'] == True]['formation_energy_per_atom'].max()
print(np.mean((min_unfav_value, max_fav_value)))

In [ ]:
# Descriptors
distrib = getDescriptors(base_structures,
                         elements_list_dict = {'U': ['S', 'Se', 'Np', 'Mo', 'W'], 
                                               'Mo': ['S', 'Se', 'Np', 'W', 'U']},
                         elements_dict = {'U': '$V_{Me}$', 'Mo': 'Mo', 'W': 'W',
                                          'Np': '$V_{X}$', 'S': 'S', 'Se': 'Se'},
                         normalize_over_cs=False)

# Print report on descriptors the heatmap with the mask and correct aspect ratio
print('distrib\'s shape', distrib.shape)
print(distrib.groupby(['FE ranges, eV/atom', 'favorability']).size())

col_subset=sorted(distrib.columns[:-5])
print('feature set: ', col_subset)
print('inequivalent features: ', len(col_subset))
print('unique sets among distrib\'s entries', sum(~distrib.duplicated(subset=col_subset)))

corr = distrib[col_subset].corr()

# Draw the heatmap with the mask and correct aspect ratio
plt.figure(figsize=(6, 6))
sns.heatmap(corr, center=0, cmap='vlag', square=True, linewidths=.5, cbar_kws={"shrink": .5})
plt.show()

# Fit descriptor-based models
X = distrib[col_subset]
y = distrib['formation_energy_per_atom']
y_model = distrib[model]
rr_dft.fit(X, y)
rr_gnn.fit(X, y_model)
rfr_dft.fit(X, y)
rfr_gnn.fit(X, y_model)

In [ ]:
print('**** DFT results ****')
print(f'Ridge Regression: RMSE = {1000 * root_mse(y, rr_dft.predict(X)):.2f} meV/atom')
print(f'Random Forest:    RMSE = {1000 * root_mse(y, rfr_dft.predict(X)):.2f} meV/atom')
print('\n**** GNN predictions ****')
print(f'Ridge Regression: RMSE = {1000 * root_mse(y, rr_gnn.predict(X)):.2f} meV/atom')
print(f'Random Forest:    RMSE = {1000 * root_mse(y, rfr_gnn.predict(X)):.2f} meV/atom')

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=2, sharey=True, figsize=(6, 3.5))
fig.suptitle(re.sub('[\d]', lambda x: f'$_{x.group()}$', formula), y=0.94)
axs[0].scatter(rr_dft.best_estimator_.coef_, range(len(col_subset)), alpha=0.7)
axs[0].scatter(rr_gnn.best_estimator_.coef_, range(len(col_subset)), alpha=0.5)
axs[1].barh(range(len(col_subset)), rfr_dft.best_estimator_.feature_importances_, height=0.4, align='edge', alpha=0.7, label='DFT')
axs[1].barh(range(len(col_subset)), rfr_gnn.best_estimator_.feature_importances_, height=-0.4, align='edge', alpha=0.4, label=f'GNN')
axs[0].set_xlim((-.03,.03))
axs[0].set_xlabel('RR coefficients, eV/pair')
axs[1].set_xscale('log')
axs[1].set_xlabel('RF feature importance, a.u.')
axs[1].set_xlim((5e-3,1))
axs[1].legend(title='$E_{F}$ values')

for i in range(2):
    axs[i].grid(alpha=0.3)
plt.yticks(list(range(len(col_subset))), col_subset)
plt.tight_layout()
plt.show()